# 19 — Learning-to-rank : optimiser l'AP en DIRECT

Insight du brief : l'AP récompense le **classement** des fraudes en haut. Or tous nos modèles
optimisent une perte de **classification** (logloss), pas le rang. XGBoost `rank:map` optimise
**directement la Mean Average Precision** = notre métrique. Jamais testé.

Features = celles du 08. Réf 08 (CatBoost classif) : **recent2 0.3662 | last 0.3607**.
Critère = recent2. Boussole : LB ≈ last − 0.004.

In [ ]:
%load_ext autoreload
%autoreload 2
import sys
from pathlib import Path
ROOT = Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import numpy as np, pandas as pd
import xgboost as xgb
from src import config as C
from src.validation import time_folds, evaluate_ap
from src.utils import op03_mask, seed_everything, make_submission
from src.features.temporal import balance_features, recency_features
from src.features.behavioral import behavioral_features
from src.encoding import oof_target_encode_train, fit_target_map, apply_target_map, recent_target_rate
seed_everything(42)
DATA = ROOT / "data"
train = pd.read_csv(DATA / "train.csv"); test = pd.read_csv(DATA / "test.csv")
sample = pd.read_csv(DATA / "sample_submission.csv")
op03 = op03_mask(train).to_numpy(); y_all = train[C.TARGET].to_numpy()
folds_full = list(time_folds(train[C.PERIOD]))
print("xgboost", xgb.__version__)

In [ ]:
EPS = 1e-6; WINDOWS = (5, 10, 20); SMOOTHING = 30
def row_features(df):
    f = pd.DataFrame(index=df.index)
    f["amount_log1p"] = np.log1p(np.maximum(df[C.AMOUNT], 0))
    f["amount_vs_origin_before"] = df[C.AMOUNT] / (np.abs(df[C.ORIGIN_BAL_BEFORE]) + EPS)
    f["amount_vs_dest_before"] = df[C.AMOUNT] / (np.abs(df[C.DEST_BAL_BEFORE]) + EPS)
    f["origin_balance_before"] = df[C.ORIGIN_BAL_BEFORE]; f["dest_balance_before"] = df[C.DEST_BAL_BEFORE]
    return pd.concat([f, balance_features(df)], axis=1)
def add_freq(X, src_df, ref_df):
    X = X.copy()
    for col in [C.ORIGIN_ACCT, C.DEST_ACCT]:
        freq = ref_df[col].value_counts(normalize=True)
        X[f"freq_{col}"] = src_df[col].map(freq).fillna(0).values
    return X
def base_build(df, ref):
    X = row_features(df).reset_index(drop=True)
    X = add_freq(X, df.reset_index(drop=True), ref)
    beh = behavioral_features(df, ref).reset_index(drop=True)
    rec = recency_features(df, ref).reset_index(drop=True)
    rt = recent_target_rate(df, ref, C.ORIGIN_ACCT, C.PERIOD, C.TARGET, WINDOWS).reset_index(drop=True)
    return pd.concat([X, beh, rec, rt], axis=1)
def feats_train(df, ref):
    X = base_build(df, ref); X["te_origin"] = oof_target_encode_train(ref, C.ORIGIN_ACCT, C.TARGET, smoothing=SMOOTHING); return X
def feats_apply(df, ref):
    X = base_build(df, ref); mp, gm = fit_target_map(ref, C.ORIGIN_ACCT, C.TARGET, smoothing=SMOOTHING)
    X["te_origin"] = apply_target_map(df, C.ORIGIN_ACCT, mp, gm); return X

def train_rank(Xtr, ytr, Xva, objective, rounds=500):
    """XGBoost en objectif de rang : un seul groupe = optimise le classement global (AP)."""
    dtr = xgb.DMatrix(Xtr, label=ytr); dtr.set_group([len(Xtr)])
    dva = xgb.DMatrix(Xva)
    params = {"objective": objective, "eta": 0.05, "max_depth": 6, "subsample": 0.8,
              "colsample_bytree": 0.8, "min_child_weight": 5, "seed": 42, "tree_method": "hist"}
    bst = xgb.train(params, dtr, num_boost_round=rounds)
    return bst.predict(dva)

## CV : objectif rang (rank:map, rank:pairwise) vs réf 08

In [ ]:
oof = {"rank:map": np.zeros(len(train)), "rank:pairwise": np.zeros(len(train))}
for tr_idx, va_idx in folds_full:
    tr_op = tr_idx[op03[tr_idx]]; va_op = va_idx[op03[va_idx]]; ref = train.iloc[tr_op]
    Xtr = feats_train(train.iloc[tr_op], ref); Xva = feats_apply(train.iloc[va_op], ref); yt = y_all[tr_op]
    for obj in oof:
        oof[obj][va_op] = train_rank(Xtr, yt, Xva, obj)
    print("fold ok")

for obj, o in oof.items():
    pf = [evaluate_ap(y_all[va[op03[va]]], o[va[op03[va]]]) for _, va in folds_full]
    print(f"{obj:16s} recent2 {np.mean(pf[-2:]):.4f} | last {pf[-1]:.4f}" + ("  <-- bat 08" if np.mean(pf[-2:]) > 0.3662 else ""))
print("\nRéf 08 (CatBoost classif) : recent2 0.3662 | last 0.3607")

## Soumission (si un objectif rang bat recent2 0.3662)

In [ ]:
BEST_OBJ = "rank:map"   # mettre l'objectif gagnant
ref_full = train.iloc[np.where(op03)[0]]; yf = y_all[op03]
Xf = feats_train(ref_full, ref_full)
te_op = op03_mask(test).to_numpy(); test_op = test.iloc[np.where(te_op)[0]]
Xte = feats_apply(test_op, ref_full)
score = train_rank(Xf, yf, Xte, BEST_OBJ)
# rank -> [0,1] pour le format de soumission (l'AP est invariante par transfo monotone)
p = (np.argsort(np.argsort(score)) / (len(score) - 1))
full = np.zeros(len(test)); full[te_op] = p
path = make_submission(test[C.ID], full, "19_rank_" + BEST_OBJ.split(':')[1])
sub = pd.read_csv(path)
assert list(sub.columns) == ["id", "target"] and len(sub) == len(test)
assert set(sub["id"]) == set(sample["id"]) and sub["target"].between(0, 1).all()
print("soumission écrite :", path, "| proba>0 :", int((sub['target'] > 0).sum()))